# 03 — Benchmark: workers × workload

A parameterized in-notebook benchmark of the worker pool: each cell of the
matrix maps `CHUNK_COUNT` prime-counting chunks across a pool of *n*
workers and records wall-clock time. The 1-worker pool is the serial
baseline, so speedups compare like against like — same dispatch path, same
warm workers.

Defaults are sized to finish well under a minute in-browser; raise `REPS`
or add workloads/worker counts for steadier numbers.

> **Environment requirements** — same as the other notebooks: cross-origin
> isolation (COOP/COEP headers, provided by `npm run serve:lite`),
> nested-worker support, and network access to the Pyodide CDN. Timings
> depend on your machine: with fewer than 4 hardware cores the 4-worker
> column will flatten out.

## Parameters — edit me

In [ ]:
WORKER_COUNTS = [1, 2, 4]
WORKLOADS = {
    "primes < 500k": 500_000,
    "primes < 1M": 1_000_000,
}
CHUNK_COUNT = 8
REPS = 1  # timed repetitions per matrix cell (median reported)

EXPECTED = {500_000: 41_538, 1_000_000: 78_498}  # π(N) correctness checks

## Setup

Install the wheel and define the workload (the same trial-division prime
counter as the other notebooks).

In [ ]:
import piplite

await piplite.install("/files/wheels/pyodide_pool-0.1.0-py3-none-any.whl")

In [ ]:
def count_primes(lo, hi):
    count = 0
    for n in range(lo, hi):
        if n < 2:
            continue
        if n == 2:
            count += 1
            continue
        if n % 2 == 0:
            continue
        d = 3
        is_prime = True
        while d * d <= n:
            if n % d == 0:
                is_prime = False
                break
            d += 2
        if is_prime:
            count += 1
    return count


def make_chunks(start, end, count):
    width = -(-(end - start) // count)  # ceiling division
    return [(lo, min(lo + width, end)) for lo in range(start, end, width)]

## Run the matrix

One pool per worker count, warmed before timing (all interpreters booted in
parallel) so the numbers measure compute + dispatch, not Pyodide boot. Each
pool's workers are terminated after its column to free memory; `REPS` timed
runs per cell, median reported.

In [ ]:
import asyncio
import statistics
import time

import pyodide_pool


async def time_map(pool, range_end):
    chunks = make_chunks(2, range_end, CHUNK_COUNT)
    times = []
    for _ in range(REPS):
        t0 = time.perf_counter()
        counts = await asyncio.gather(
            *(pool.submit(count_primes, lo, hi) for lo, hi in chunks)
        )
        times.append(time.perf_counter() - t0)
        total = sum(counts)
        assert total == EXPECTED[range_end], (range_end, total)
    return statistics.median(times)


results = {}  # (workload name, worker count) -> median seconds
for workers in WORKER_COUNTS:
    pool = await pyodide_pool.create_pool(pool_size=workers)
    await asyncio.gather(*(pool.submit(lambda i=i: i) for i in range(workers)))
    for name, range_end in WORKLOADS.items():
        seconds = results[(name, workers)] = await time_map(pool, range_end)
        print(f"{name} × {workers} worker(s): {seconds:.2f} s")
    pool.terminate()

## Results

Median wall-clock per cell, with speedup vs the 1-worker column in
parentheses.

In [ ]:
from IPython.display import Markdown

header = "| workload | " + " | ".join(f"{n} worker(s)" for n in WORKER_COUNTS) + " |"
rule = "| --- |" + " ---: |" * len(WORKER_COUNTS)
rows = []
for name in WORKLOADS:
    base = results[(name, WORKER_COUNTS[0])]
    cells = [
        f"{results[(name, n)]:.2f} s ({base / results[(name, n)]:.2f}×)"
        for n in WORKER_COUNTS
    ]
    rows.append("| " + name + " | " + " | ".join(cells) + " |")
Markdown("\n".join([header, rule, *rows]))

## Speedup chart

(The first `matplotlib` import downloads the package into the kernel — give
it a few seconds.)

In [ ]:
import matplotlib.pyplot as plt

colors = ["#2a78d6", "#008300"]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(
    WORKER_COUNTS,
    WORKER_COUNTS,
    linestyle="--",
    linewidth=1.5,
    color="#b3b2ab",
    label="ideal",
    zorder=1,
)
for color, name in zip(colors, WORKLOADS):
    base = results[(name, WORKER_COUNTS[0])]
    speedups = [base / results[(name, n)] for n in WORKER_COUNTS]
    ax.plot(
        WORKER_COUNTS,
        speedups,
        marker="o",
        markersize=7,
        linewidth=2,
        color=color,
        label=name,
    )
ax.set_xlabel("workers")
ax.set_ylabel("speedup vs 1 worker")
ax.set_xticks(WORKER_COUNTS)
ax.set_title("Worker-pool speedup")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
ax.grid(axis="y", color="#e3e2dc", linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False)
plt.show()

## Success marker

In [ ]:
print("BENCH_DEMO_OK")